# Search Notebook for Dialysis & Transplant Data

## Search for:
- Dialysis within OMOP
    - Dialysis Access details suplimented from Elastic
- Transplants within OMOP

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

import math

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

## Extract Dialysis & Transplot Activity from OMOP

### Load in Patients of Interest

In [ ]:
inclusion_patients_df = pd.read_csv(raw_data_path+"elasticsearch_search_hits/chronic_kidney_disease_inclusion_patients.csv")[['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']]

### Dialysis Activity

In [ ]:
with duckdb.connect() as conn:
    renal_dialysis_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                            SELECT DISTINCT po.master_person_id
                                                   , ip.patient_identifier3
                                                   , ip.patient_identifier4
                                                   , ip.patient_identifier2
                                                   , vo.appointment_id
                                                   , vo.encounter_id
                                                   , c2.concept_name AS visit_concept_name
                                                   , c1.concept_name AS procedure_concept_name
                                                   , po.procedure_source_value AS procedure_opcs_code
                                                   , po.procedure_datetime
                                                   , po.procedure_end_datetime
                                                   , CASE WHEN UPPER(po.source_table_provenance) LIKE '%EPIC%' THEN 'EPIC'
                                                          WHEN UPPER(c2.concept_name) LIKE '%INPATIENT%' THEN 'Inpatient'
                                                          ELSE 'Outpatient' END AS activity_identifier
                                            FROM ext_procedure_occurrence AS po
                                                INNER JOIN inclusion_patients_df AS ip
                                                    ON po.master_person_id = ip.master_person_id
                                                LEFT JOIN concept AS c1
                                                    ON po.procedure_concept_id = c1.concept_id
                                                LEFT JOIN ext_visit_occurrence AS vo
                                                    ON po.master_visit_occurrence_id = vo.master_visit_occurrence_id
                                                LEFT JOIN concept AS c2
                                                    ON vo.visit_concept_id = c2.concept_id
                                            WHERE UPPER(c1.concept_name) LIKE '%DIALYSIS%'
                                            ORDER BY po.master_person_id, po.procedure_datetime;""").df()

In [ ]:
renal_dialysis_df.head()

In [ ]:
renal_dialysis_df.to_csv(raw_data_path+'elasticsearch_search_hits/raw_dialysis_activity.csv', index=False)

### Transplant Activity

In [ ]:
with duckdb.connect() as conn:
    transplants_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                            SELECT DISTINCT po.master_person_id
                                                   , ip.patient_identifier3
                                                   , ip.patient_identifier4
                                                   , ip.patient_identifier2
                                                   , vo.appointment_id
                                                   , vo.encounter_id
                                                   , c2.concept_name AS visit_concept_name
                                                   , c1.concept_name AS procedure_concept_name
                                                   , po.procedure_source_value AS procedure_opcs_code
                                                   , po.procedure_datetime
                                                   , po.procedure_end_datetime
                                                   , CASE WHEN UPPER(po.source_table_provenance) LIKE '%EPIC%' THEN 'EPIC'
                                                          WHEN UPPER(c2.concept_name) LIKE '%INPATIENT%' THEN 'Inpatient'
                                                          ELSE 'Outpatient' END AS activity_identifier
                                            FROM ext_procedure_occurrence AS po
                                                INNER JOIN inclusion_patients_df AS ip
                                                    ON po.master_person_id = ip.master_person_id
                                                LEFT JOIN concept AS c1
                                                    ON po.procedure_concept_id = c1.concept_id
                                                LEFT JOIN ext_visit_occurrence AS vo
                                                    ON po.master_visit_occurrence_id = vo.master_visit_occurrence_id
                                                LEFT JOIN concept AS c2
                                                    ON vo.visit_concept_id = c2.concept_id
                                            WHERE UPPER(c1.concept_name) LIKE '%ALLOTRANSPLANT%' 
                                            ORDER BY po.master_person_id, po.procedure_datetime;""").df()

In [ ]:
transplants_df.head()

In [ ]:
transplants_df.to_csv(raw_data_path+'elasticsearch_search_hits/raw_transplant_activity.csv', index=False)

### eGFR Results <= 15

#### OMOP Results

In [ ]:
with duckdb.connect() as conn:
    omop_gfr_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                       SELECT DISTINCT ip.master_person_id
                                              , m.min_measurement_date
                                        FROM inclusion_patients_df AS ip
                                            INNER JOIN (SELECT DISTINCT master_person_id
                                                              , min(measurement_date) AS min_measurement_date
                                                       FROM ext_measurement
                                                       WHERE UPPER(measurement_source_value_name) LIKE '%GFR%' AND value_as_number <= 15
                                                       GROUP BY master_person_id) as m
                                                using(master_person_id)
                                        ;""").df()

#### Epic Data

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

In [ ]:
gstt_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            gstt_number.append(i)

In [ ]:
nhs_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                nhs_number.append(int(i))
            except:
                pass

In [ ]:
epic_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            epic_number.append(int(i))

In [ ]:
# --- Explore Fields in a Given Index ---
index = 'lab_results'
columns = ['patient_identifier1', 'patient_activity_document_identifiers', 'patient_identifier2', 'patient_identifier3',
           'patient_EpicId', 'document_Name', 'document_CreatedWhen', 'document_Fields']

In [ ]:
epic_egfr_df = pd.DataFrame()
results = 0

for i in range(math.ceil(len(gstt_number)/65536)):
    if i == 0:
        gstt_number_list = gstt_number[:65536]
    else:
        gstt_number_list = gstt_number[65536:]
    
    gfr_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_identifier1": gstt_number_list }},
                                { "terms": { "patient_identifier3": nhs_number }},
                                { "terms": { "patient_identifier2": epic_number }}
                                ],
                            "minimum_should_match": 1
                            }
                        },
                    {
                        "nested": {
                            "path": "document_Fields",
                            "query": {
                                "bool": {
                                    "should": [
                                        {
                                            "terms": {
                                                "document_Fields.label": [
                                                    "eGFR",
                                                    "eGFR by CKD-EPI (2009)",
                                                    "eGFR by MDRD",
                                                    "EGFR BY CKD -15 MINUTES"
                                                    ]
                                                }
                                            }
                                        ],
                                    "minimum_should_match": 1
                                    }
                                },
                            "score_mode": "none"
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=gfr_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(df['_id'].to_list())]
            temp_df = temp_df[~temp_df['document_Name'].isin(docs_to_remove)]
        except Exception as e:
            print(f"Deduplication error: {e}")
    
        epic_egfr_df = pd.concat([epic_egfr_df, temp_df], ignore_index=True)
        #epic_egfr_df = epic_egfr_df.drop_duplicates()

    del temp_df
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {epic_egfr_df.shape[0]:,}\n")

In [ ]:
epic_egfr_df.head()

In [ ]:
nhs_number_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower(): 
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            nhs_number_dict[nhs_num]=row[0]

epic_egfr_df['masterPersonId_nhs'] = epic_egfr_df['patient_identifier3'].map(nhs_number_dict)

In [ ]:
gstt_num_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            gstt_num_dict[nhs_num]=row[0]

def gstt_number(row):
    master_person_id = ''
    try:
        for num in row:
            if (num) in gstt_num_dict.keys():
                master_person_id = gstt_num_dict[num]
    except:
        pass

    return master_person_id

epic_egfr_df['masterPersonId_gstt'] = epic_egfr_df['patient_identifier1'].apply(gstt_number)

In [ ]:
epic_mrn_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            epic_mrn_dict[nhs_num]=row[0]

epic_egfr_df['masterPersonId_mrn'] = epic_egfr_df['patient_identifier2'].map(epic_mrn_dict)

In [ ]:
epic_egfr_df.insert(0, 'master_person_id', epic_egfr_df['masterPersonId_nhs'].combine_first(epic_egfr_df['masterPersonId_mrn']).combine_first(epic_egfr_df['masterPersonId_gstt']))

In [ ]:
cols = ['master_person_id', 'resultId', 'resultDate', 'label', 'valueNum', 'unitOfMeasure']

rename_dict = {'resultDate' : 'measurement_datetime', 'label' : 'measurement_source_value_name', 'valueNum' : 'value_as_number', 'unitOfMeasure' : 'unit_source_value'}

epic_egfr_df_exploded = epic_egfr_df.explode('document_Fields').reset_index(drop=True)

epic_egfr_results_df = pd.concat(
    [
        epic_egfr_df_exploded.drop(columns=['document_Fields']),
        epic_egfr_df_exploded['document_Fields'].apply(pd.Series)
    ],
    axis=1
)

rel_tests = ['eGFR by CKD-EPI (2009)', 'eGFR', 'eGFR by MDRD']

epic_egfr_results_df = epic_egfr_results_df[(epic_egfr_results_df['label'].isin(rel_tests))&(epic_egfr_results_df['valueNum'].notna())&(epic_egfr_results_df['valueNum']<=15)][cols].rename(columns=rename_dict).reset_index(drop=True)

del epic_egfr_df, epic_egfr_df_exploded, rel_tests, rename_dict, epic_mrn_dict, gstt_num_dict, nhs_number_dict, index, columns, gstt_number, nhs_number, epic_number, gfr_query, es

epic_egfr_results_df['measurement_datetime'] = pd.to_datetime(epic_egfr_results_df['measurement_datetime'])
epic_egfr_results_df['measurement_date'] = epic_egfr_results_df['measurement_datetime'].dt.date

epic_egfr_results_df = epic_egfr_results_df.sort_values(by=['master_person_id', 'measurement_datetime']).groupby('master_person_id').first().reset_index()

epic_egfr_results_df = epic_egfr_results_df[['master_person_id', 'measurement_date']].rename(columns={'measurement_date': 'min_measurement_date'})

In [ ]:
gfr_df = pd.concat([omop_gfr_df, epic_egfr_results_df])

del omop_gfr_df, epic_egfr_results_df

gfr_df['min_measurement_date'] = pd.to_datetime(gfr_df['min_measurement_date']).dt.date

gfr_df = gfr_df.sort_values(by=['master_person_id', 'min_measurement_date']).groupby('master_person_id').first().reset_index()

gfr_df.head()

In [ ]:
gfr_df.to_csv(raw_data_path+'elasticsearch_search_hits/first_gfr_below_15.csv', index=False)

## Search Elastic for Dialysis Access

### Import Dialysis Activity Data

In [ ]:
renal_dialysis_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/raw_dialysis_activity.csv')

renal_dialysis_df['appointment_id'] = renal_dialysis_df['appointment_id'].apply(lambda x: str(x)[:-2])
renal_dialysis_df['encounter_id'] = renal_dialysis_df['encounter_id'].apply(lambda x: str(x)[:-2])

renal_dialysis_df['procedure_datetime'] = pd.to_datetime(renal_dialysis_df['procedure_datetime'])

renal_dialysis_df.head()

#### encounter_id vs appointment_id
- For EPIC data, `encounter_id` is the EPIC CSN, and can be used to join to activity data in Elastic
- For Legacy data, `appointment_id` and `encounter_id` are interchangable where present, though `encounter_id` looks to be better populated

### Get EPIC CSNs

In [ ]:
epiccsn_list = list(set(renal_dialysis_df[(renal_dialysis_df['activity_identifier']=='EPIC')&(renal_dialysis_df['encounter_id'].notna())]['encounter_id'].to_list()))

### Get Legacy In- and Out-Patient Activity Numbers

In [ ]:
vistnum_list = []

for encounter_id in renal_dialysis_df[(renal_dialysis_df['activity_identifier']=='Inpatient')&(renal_dialysis_df['encounter_id'].notna())]['encounter_id']:
    vistnum_list.append('IP'+str(encounter_id))

vistnum_list = list(set(vistnum_list))

In [ ]:
for appointment_id in renal_dialysis_df[(renal_dialysis_df['activity_identifier']=='Inpatient')&(renal_dialysis_df['appointment_id'].notna())]['appointment_id']:
    vistnum_list.append('IP'+str(appointment_id))

vistnum_list = list(set(vistnum_list))

In [ ]:
for appointment_id in renal_dialysis_df[(renal_dialysis_df['activity_identifier']=='Outpatient')&(renal_dialysis_df['encounter_id'].notna())]['encounter_id']:
    vistnum_list.append('OP'+str(encounter_id))

vistnum_list = list(set(vistnum_list))

In [ ]:
for appointment_id in renal_dialysis_df[(renal_dialysis_df['activity_identifier']=='Outpatient')&(renal_dialysis_df['appointment_id'].notna())]['appointment_id']:
    vistnum_list.append('OP'+str(appointment_id))

vistnum_list = list(set(vistnum_list))

### Initialise

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

In [ ]:
# --- Explore Indices ---
print("🔍 Available Indices:")
for i in list(es.indices.get_alias().keys()):
    print("-", i)

In [ ]:
# --- Explore Fields in a Given Index ---
index = [
      'observations'
    , 'orders'
    , 'notes'
        ]  # 🔧 Set your index name here

for idx in index:
    print(f"\n📑 Fields in index: {idx}")
    mapping = get_field_mapping(es, idx)
    for field in mapping[idx]["mappings"]["properties"]:
        print("-", field)

In [ ]:
# --- Define Query Parameters ---
# 🔧 List of fields you'd like to return (or leave empty to return all)
columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'patient_identifier1',
           'activity_Date', 'activity_VisitNumber', 'activity_EncounterEpicCsn', 'activity_VisitService', 'activity_Type',
           'document_CreatedWhen', 'document_Name', 'document_OrderName', 'document_Format', 'document_Content']

In [ ]:
n = 65536

In [ ]:
access_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(vistnum_list, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "document_Content : (\"temporary\" OR \"tunnelled\" OR \"fistula\" OR \"PD\" OR \"line\" OR \"access\" OR \"Type/Side\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"activity_VisitNumber": sub_list}}
                            ],
                            "minimum_should_match": 1        
                        }
                    }
                ]
            }
        }
    }

    temp_df = es_docs_to_df(es, index=index, query=query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(access_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        access_df = pd.concat([access_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {access_df.shape[0]:,}\n")

    i+=1

In [ ]:
i = 1

full_list = list_chunker(epiccsn_list, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "document_Content : (\"temporary\" OR \"tunnelled\" OR \"fistula\" OR \"PD\" OR \"line\" OR \"access\" OR \"Type/Side\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                {"terms": {"activity_EncounterEpicCsn": sub_list}}
                            ],
                            "minimum_should_match": 1        
                        }
                    }
                ]
            }
        }
    }

    temp_df = es_docs_to_df(es, index=index, query=query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(access_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        access_df = pd.concat([access_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {access_df.shape[0]:,}\n")
    
    i += 1

In [ ]:
access_df = access_df[access_df['document_Content'].notna()].reset_index(drop=True)

access_df.head()

In [ ]:
access_df.shape

In [ ]:
access_df.to_csv(raw_data_path+'elasticsearch_search_hits/access_search_hits.csv', index=False)

## Extract Relevant Dialysis Access

#### Import Dialysis Access Details

In [ ]:
access_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/access_search_hits.csv')

access_df = access_df.drop(columns=['document_OrderName', 'document_Format', '_score'])

access_df['patient_identifier2'] = access_df['patient_identifier2'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
access_df['patient_identifier3'] = access_df['patient_identifier3'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
access_df['activity_EncounterEpicCsn'] = access_df['activity_EncounterEpicCsn'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])

access_df.insert(7, 'activty_Identifier', access_df['activity_VisitNumber'].combine_first(access_df['activity_EncounterEpicCsn']))

access_df.head()

### Clean Data

In [ ]:
def access_string_regex(text):
    if pd.isna(text) or text is None:
        return np.NaN

    text = str(text)

    pattern1 = r'''Type/Side(.*?)Type/?'''

    pattern2 = r'''Type/Side(.*?)Site[\t\n\r\f\v]'''

    pattern3 = r'''Site(.*?)[QTC]'''

    pattern4 = r'''Vascular Access(.*?)A'''

    pattern5 = r'''Vascular Access(.*)$'''

    pattern6 = r'''Type/Side(.*?)(?:Site|Plan|Weight|Transonic|Dominant|MR.|Sitting|Lock|Check|Prescribing)'''

    pattern7 = r'''Type/Side(.*)$'''

    pattern8 = r'''Site(.*?)(?:Site|Plan|Weight|Transonic|Dominant|MR.|Sitting|Lock|Check|Prescribing)'''

    pattern9 = r'''Type/Side(.*?)\n{2,}'''

    pattern10 = r'''Site(.*?)\n{2,}'''

    all_matches = []

    for pattern in [pattern1, pattern2, pattern3, pattern4, pattern5, pattern6, pattern7, pattern8, pattern9, pattern10
                   ]:
        regex = re.compile(pattern, re.IGNORECASE|re.DOTALL)
        matches = regex.findall(text)

        all_matches.extend(matches)
    
    cleaned_matches = []
    seen = set()
    
    for match in all_matches:
        clean_match = re.sub(r'\s+', ' ', match)
        clean_match = clean_match.title()
        if clean_match and clean_match not in seen and len(clean_match) > 4 and len(clean_match) < 50:
            cleaned_matches.append(clean_match)
            seen.add(clean_match)

    if len(cleaned_matches) == 1:
        return cleaned_matches[0]
    elif len(cleaned_matches) == 0:
        return np.nan
    else:
        return ', '.join(cleaned_matches)

In [ ]:
access_df['access'] = access_df['document_Content'].apply(lambda x: access_string_regex(x))

### Extract Access Specifics

In [ ]:
cols = ['activty_Identifier', 'activity_Date', 'access']
docs = ['Prescription/Flowchart', 'Hemodialysis Outpatient']

access_refined_df = access_df[(access_df['access'].notna())&(access_df['document_Name'].isin(docs))][cols].reset_index(drop=True)

del access_df

access_refined_df.head()

In [ ]:
def rel_access_det(row):
    proper_list = []

    for val in row['access']:
        text = val.lower()
    
        if 'tunn' in text or 'fist' in text or 'central' in text:
            proper_list.append(val)

    return proper_list

In [ ]:
def extract_contents(lst):
    return ', '.join(map(str, lst))

In [ ]:
access_details_df = access_refined_df['access'].str.findall(r'((?:\w*-)?(?:\w* -?){1,}\[\d{2,}\]|A:\s\w*.?\w*)').reset_index()

access_details_df['access_details'] = access_details_df.apply(lambda x: rel_access_det(x), axis=1).apply(extract_contents)

access_details_df = access_details_df.drop(columns=['index', 'access'])

In [ ]:
full_access_details_df = access_refined_df.join(access_details_df)

del access_details_df, access_refined_df

full_access_details_df = full_access_details_df.drop(columns='access')

full_access_details_df.head()

In [ ]:
full_access_details_df.to_csv(raw_data_path+'elasticsearch_search_hits/20251023_access_details.csv', index=False)

## Create Endpoint Data

#### Endpoints:
- First Dialysis Date
- Unless there's a transplant before first dialysis
- First GFR <=15 Date
- AKI Activity
- Death

#### Import Transplant Activity Detials

In [ ]:
cols = ['master_person_id', 'appointment_id', 'encounter_id', 'procedure_datetime', 'procedure_opcs_code', 'procedure_concept_name']

transplants_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/raw_transplant_activity.csv')

transplants_df['appointment_id'] = transplants_df['appointment_id'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
transplants_df['encounter_id'] = transplants_df['encounter_id'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
transplants_df['procedure_datetime'] = pd.to_datetime(transplants_df['procedure_datetime']).dt.date

transplants_df = transplants_df[cols].drop_duplicates().reset_index(drop=True)

transplants_df.head()

#### Import Dialysis Activity Details

In [ ]:
cols = ['master_person_id', 'appointment_id', 'encounter_id', 'procedure_datetime', 'procedure_opcs_code', 'procedure_concept_name']

renal_dialysis_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/raw_dialysis_activity.csv')

renal_dialysis_df['appointment_id'] = renal_dialysis_df['appointment_id'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
renal_dialysis_df['encounter_id'] = renal_dialysis_df['encounter_id'].apply(lambda x : x if str(x)=='nan' else str(x)[:-2])
renal_dialysis_df['procedure_datetime'] = pd.to_datetime(renal_dialysis_df['procedure_datetime']).dt.date

renal_dialysis_df = renal_dialysis_df[cols].drop_duplicates().reset_index(drop=True)

renal_dialysis_df.head()

#### Import Access Details

In [ ]:
full_access_details_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/20251023_access_details.csv')
full_access_details_df.insert(0, 'appointment_id', full_access_details_df['activty_Identifier'].apply(lambda x: int(str(x)[2:]) if str(x)[:2]=='IP' or str(x)[:2]=='OP' else x))

full_access_details_df.drop(columns=['activty_Identifier'], inplace=True)
full_access_details_df['activity_Date'] = pd.to_datetime(full_access_details_df['activity_Date']).dt.date
full_access_details_df = full_access_details_df.drop_duplicates().reset_index(drop=True)

full_access_details_df.head()

#### Import GFR Data

In [ ]:
gfr_df = pd.read_csv(raw_data_path+'elasticsearch_search_hits/first_gfr_below_15.csv')

gfr_df['min_measurement_date'] = pd.to_datetime(gfr_df['min_measurement_date']).dt.date
gfr_df = gfr_df.rename(columns={'min_measurement_date': 'first_gfr_15_date'})

gfr_df.head()

#### Import AKI Activity Data

In [ ]:
col = ['master_person_id', 'condition_start_date', 'condition_icd10_code', 'condition_concept_name']

aki_refined_activity_df = pd.read_csv(raw_data_path+"20251203_aki_activity_search_results.csv")[col]

aki_refined_activity_df['condition_start_date'] = pd.to_datetime(aki_refined_activity_df['condition_start_date']).dt.date
aki_refined_activity_df = aki_refined_activity_df.sort_values(['master_person_id', 'condition_start_date']).groupby(['master_person_id']).first().reset_index()

aki_refined_activity_df.head()

### Combine Dialysis & Access Details

In [ ]:
first_dialysis_df = renal_dialysis_df.merge(full_access_details_df, how='left', on='appointment_id', suffixes=('', '_x')).merge(full_access_details_df, how='left', left_on='encounter_id', right_on='appointment_id', suffixes=('', '_y'))

first_dialysis_df['activity_Date'] = first_dialysis_df['activity_Date'].combine_first(first_dialysis_df['activity_Date_y'])
first_dialysis_df['access_details'] = first_dialysis_df['access_details'].combine_first(first_dialysis_df['access_details_y'])

del first_dialysis_df['appointment_id_y'], first_dialysis_df['activity_Date_y'], first_dialysis_df['access_details_y']

first_dialysis_df = first_dialysis_df.drop_duplicates().sort_values(by=['master_person_id', 'procedure_datetime']).groupby('master_person_id').first().reset_index()

In [ ]:
def diff_as_int(x):
    text = str(x)

    value = 0

    if text=='nan':
        value = np.NAN
    elif text=='0:00:00':
        value = 0
    else:
        value = int(re.findall(r'(.*)\sday', text)[0])

    return abs(value)

In [ ]:
first_dialysis_df['Diff'] = first_dialysis_df['activity_Date']-first_dialysis_df['procedure_datetime']
first_dialysis_df['Diff'] = first_dialysis_df['Diff'].apply(lambda x: diff_as_int(x))

first_dialysis_df['access_details'] = first_dialysis_df.apply(lambda x: None if x['Diff']>=2 else x['access_details'], axis=1)

first_dialysis_df = first_dialysis_df.drop(columns=['Diff', 'activity_Date'])

In [ ]:
first_transplants_df = transplants_df.drop_duplicates().sort_values(by=['master_person_id', 'procedure_datetime']).groupby('master_person_id').first().reset_index()

In [ ]:
rename_dict = {'procedure_datetime': 'endpoint_date',
               'procedure_opcs_code': 'endpoint_opcs_code',
               'procedure_concept_name': 'endpoint_description'
              }

In [ ]:
endpoints_df = pd.concat([first_dialysis_df, first_transplants_df])

endpoints_df = endpoints_df.drop_duplicates().sort_values(by=['master_person_id', 'procedure_datetime']).groupby('master_person_id').first().reset_index()
endpoints_df.insert(3, 'endpoint_type', endpoints_df['procedure_concept_name'].apply(lambda x: 'Dialysis' if 'dialysis' in x.lower() else 'Transplant'))

endpoints_df = endpoints_df.rename(columns=rename_dict)

endpoints_df.head()

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2', 'addl_pat_flg', 'dateOfBirth',
        'deathOfDate', 'gender', 'raceCode', 'race', 'postcode', 'imd_decile', 'imd_rank', 'measureDate']

inclusion_patients_df = pd.read_csv(raw_data_path+"elasticsearch_search_hits/chronic_kidney_disease_inclusion_patients.csv")[cols]

#### Add Dialysis & Transplant Endpoints

In [ ]:
cols = ['master_person_id', 'endpoint_type', 'endpoint_date', 'endpoint_opcs_code', 'endpoint_description', 'access_details']

inclusion_patients_refined_df = inclusion_patients_df.merge(endpoints_df[cols], how='left', on='master_person_id')
inclusion_patients_refined_df.insert(16, 'endpoint_code', inclusion_patients_refined_df['endpoint_opcs_code'])
inclusion_patients_refined_df.insert(17, 'endpoint_code_type', inclusion_patients_refined_df['endpoint_opcs_code'].apply(lambda x: np.NaN if pd.isna(x) else 'OPCS'))

#### Add GFR Endpoints

In [ ]:
inclusion_patients_refined_df = inclusion_patients_refined_df.merge(gfr_df, how='left', on='master_person_id')

inclusion_patients_refined_df['endpoint_date'] = inclusion_patients_refined_df['endpoint_date'].combine_first(inclusion_patients_refined_df['first_gfr_15_date'])
inclusion_patients_refined_df['endpoint_type'] = inclusion_patients_refined_df.apply(lambda x: 'eGFR <= 15' if x['endpoint_date']==x['first_gfr_15_date'] and pd.isna(x['endpoint_code']) else x['endpoint_type'], axis=1)

#### Add AKI Activity Endpoints

In [ ]:
inclusion_patients_refined_df = inclusion_patients_refined_df.merge(aki_refined_activity_df, how='left', on='master_person_id')

inclusion_patients_refined_df['endpoint_date'] = inclusion_patients_refined_df['endpoint_date'].combine_first(inclusion_patients_refined_df['condition_start_date'])
inclusion_patients_refined_df['endpoint_type'] = inclusion_patients_refined_df.apply(lambda x: 'First AKI Episode' if x['endpoint_date']==x['condition_start_date'] and pd.isna(x['endpoint_type']) else x['endpoint_type'], axis=1)

inclusion_patients_refined_df['endpoint_code'] = inclusion_patients_refined_df['endpoint_code'].combine_first(inclusion_patients_refined_df['condition_icd10_code'])
inclusion_patients_refined_df['endpoint_code_type'] = inclusion_patients_refined_df.apply(lambda x: 'ICD10' if x['endpoint_code']==x['condition_icd10_code'] and pd.notna(x['condition_icd10_code']) else x['endpoint_code_type'], axis=1)

#### Add Death Endpoints

In [ ]:
inclusion_patients_refined_df['endpoint_date'] = inclusion_patients_refined_df['endpoint_date'].combine_first(inclusion_patients_refined_df['deathOfDate'])
inclusion_patients_refined_df['endpoint_type'] = inclusion_patients_refined_df.apply(lambda x: 'Death' if x['endpoint_date']==x['deathOfDate'] and pd.isna(x['endpoint_type']) else x['endpoint_type'], axis=1)

#### Finalise Refined Inclusion Patients DataFrame

In [ ]:
cols_to_drop =['first_gfr_15_date', 'condition_start_date', 'condition_icd10_code', 'condition_concept_name', 'endpoint_opcs_code', 'endpoint_description']

inclusion_patients_refined_df = inclusion_patients_refined_df.drop(columns=cols_to_drop).rename(columns={'finalMeasurementDate' : 'inclusion_date'})

del cols_to_drop, gfr_df, aki_refined_activity_df

inclusion_patients_refined_df.head()

In [ ]:
inclusion_patients_refined_df['endpoint_type'].value_counts(dropna=False)

## Save

In [ ]:
## Raw Data
endpoints_df.to_csv(os.path.join(raw_data_path, 'elasticsearch_search_hits/20251204_endpoints_data.csv'), index=False)

In [ ]:
## Refined Inclusion Patients Data
inclusion_patients_refined_df.to_csv(os.path.join(data_path, 'chronic_kidney_disease_refined_inclusion_patients.csv'), index=False)

## Sandbox